<a href="https://colab.research.google.com/github/vaniaklinov-crypto/Practice/blob/view/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%87%D0%B0%D1%81%D1%82%D1%8C_(1)_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#АНАЛИЗ ЗЛОНАМЕРЕННОГО ИСПОЛЬЗОВАНИЯ ИИ

# 1. УСТАНОВКА БИБЛИОТЕК
!pip install -q pandas numpy nltk scikit-learn matplotlib seaborn wordcloud folium \
               sentence-transformers umap-learn hdbscan imbalanced-learn

# 2. ИМПОРТЫ
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import joblib
import folium
from folium.plugins import HeatMap
import umap
import hdbscan
from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
import nltk
nltk.download('punkt_tab')

print("Все библиотеки установлены и импортированы")

In [ ]:
import random
from datetime import datetime, timedelta
import folium
from folium.plugins import HeatMap

In [ ]:
# 3. ЗАГРУЗКА ДАТАСЕТОВ
#Загружаем HackerNews с колонкой Link
df_hacker = pd.read_excel('/content/drive/MyDrive/TheHackerNews_Dataset.xlsx')

#Переименуем
df_hacker = df_hacker.rename(columns={'Article': 'text', 'Label': 'label'})

#Колонка link
if 'Link' not in df_hacker.columns:
    raise ValueError("Колонка 'Link' не найдена! Убедитесь, что она есть в Excel.")

print("Примеры ссылок:")
print(df_hacker['Link'].head())

In [ ]:
def extract_date_from_url(url):
    if pd.isna(url):
        return pd.NaT
    url = str(url)

    #Паттерн: /2021/08/, /2021/8/, /2021/08
    match = re.search(r'/(\d{4})/(\d{1,2})/', url)
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        #Приводим к первому числу месяца (021-08-01)
        try:
            return pd.Timestamp(year=year, month=month, day=1)
        except:
            return pd.NaT
    return pd.NaT

df_hacker['date_published'] = df_hacker['Link'].apply(extract_date_from_url)

In [ ]:
print(f"Извлечено дат из URL: {df_hacker['date_published'].notna().sum()} из {len(df_hacker)}")
print("\nПримеры:")
print(df_hacker[['Link', 'date_published']].head(10))

In [ ]:
#reports.csv
df_reports = pd.read_csv('/content/drive/MyDrive/reports.csv')
df_reports = df_reports.rename(columns={'tags': 'label'})
df_reports['date_published'] = pd.to_datetime(df_reports['date_published'], errors='coerce')
df_reports = df_reports[['text', 'label', 'date_published']]

#Объединение
df = pd.concat([
    df_hacker[['text', 'label', 'date_published']],
    df_reports
], ignore_index=True)

df = df.dropna(subset=['text']).reset_index(drop=True)
print(f"Итого строк: {len(df)}, с датой: {df['date_published'].notna().sum()}")

In [ ]:
# 4. ПРЕДОБРАБОТКА ТЕКСТА
def preprocess_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'http[s]?://\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

df['clean_text'] = df['text'].apply(preprocess_text)
df['clean_text_str'] = df['clean_text'].apply(' '.join)

print("Предобработка завершена")

In [ ]:
# 5. ФИЛЬТРАЦИЯ: ТОЛЬКО НОВОСТИ О ЗЛОНАМЕРЕННОМ ИИ

if 'is_malicious_ai' in df.columns:
    df_ai = df[df['is_malicious_ai'] == True].copy()
elif 'ai_threat' in df.columns:
    df_ai = df[df['ai_threat'] == 1].copy()
else:
    ai_keywords = ['ai', 'artificial intelligence', 'machine learning']
    threat_keywords = [
        'attack', 'weapon', 'fraud', 'hack', 'deepfake', 'drone', 'autonomous',
        'scam', 'phishing', 'infrastructure', 'risk', 'danger', 'threat'
    ]
    ai_pattern = '|'.join(ai_keywords)
    threat_pattern = '|'.join(threat_keywords)

    has_ai = df['clean_text_str'].str.contains(ai_pattern, case=False, na=False)
    has_threat = df['clean_text_str'].str.contains(threat_pattern, case=False, na=False)
    df_ai = df[has_ai & has_threat].copy()

#Убираем слишком короткие
df_ai = df_ai[df_ai['clean_text_str'].str.len() > 30].copy()

print(f"Новостей о злонамеренном ИИ: {len(df_ai)}")
print("Примеры:")
print(df_ai[['text', 'clean_text_str']].head(3))

In [ ]:
# 6. ИЗВЛЕЧЕНИЕ СТРАН — УЛУЧШЕННАЯ ВЕРСИЯ

#Расширенный словарь (синонимы, столицы, аббревиатуры)
country_keywords = {
    'USA': ['usa', 'united states', 'america', 'us', 'washington', 'dc', 'washington dc', 'u.s.', 'u.s.a'],
    'China': ['china', 'beijing', 'prc', 'chinese'],
    'Russia': ['russia', 'moscow', 'russian', 'kremlin', 'putin']
}

def extract_countries_improved(text):
    found = set()
    text_lower = text.lower()
    for country, keywords in country_keywords.items():
        if any(kw in text_lower for kw in keywords):
            found.add(country)
    return list(found) if found else None

df_ai['countries'] = df_ai['clean_text_str'].apply(extract_countries_improved)
df_exploded = df_ai.explode('countries').dropna(subset=['countries'])

print(f"Упоминаний стран: {len(df_exploded)}")
print("\nРаспределение:")
print(df_exploded['countries'].value_counts())

In [ ]:
# 7. КЛАСТЕРИЗАЦИЯ: BERT + UMAP + HDBSCAN

print("BERT эмбеддинги...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df_ai['clean_text_str'].tolist(), batch_size=64, show_progress_bar=True)

print("UMAP...")
reducer = umap.UMAP(n_neighbors=30, min_dist=0.0, n_components=5, random_state=42)
umap_emb = reducer.fit_transform(embeddings)

print("HDBSCAN...")
# Защита от ошибки: если меньше 2 кластеров — используем KMeans
if len(df_ai) < 500:
    k = min(5, len(df_ai)//100)
    clusterer = KMeans(n_clusters=k, n_init=10, random_state=42)
    df_ai['cluster'] = clusterer.fit_predict(umap_emb)
else:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min(250, len(df_ai)//10),
        min_samples=30,
        cluster_selection_epsilon=0.5
    )
    df_ai['cluster'] = clusterer.fit_predict(umap_emb)

# Проверка Silhouette
unique_labels = np.unique(df_ai['cluster'])
if len(unique_labels) > 1 and len(unique_labels) < len(df_ai):
    score = silhouette_score(umap_emb, df_ai['cluster'])
    print(f"Silhouette Score: {score:.3f}")
else:
    print("Silhouette не считается: слишком мало кластеров или данных")
    score = 0.0

# Автоимена
cluster_names = {}
for c in sorted(df_ai['cluster'].unique()):
    if c == -1:
        cluster_names[c] = "Другое"
        continue
    texts = df_ai[df_ai['cluster'] == c]['clean_text_str']
    top_words = [w for w, _ in Counter(' '.join(texts).split()).most_common(15)]
    name = "Кластер " + str(c)
    if any(w in top_words for w in ['deepfake', 'video', 'face']): name = "Deepfake & медиафальсификации"
    elif any(w in top_words for w in ['scam', 'fraud', 'phishing']): name = "Мошенничество и фишинг"
    elif any(w in top_words for w in ['military', 'drone', 'weapon']): name = "Военное применение ИИ"
    elif any(w in top_words for w in ['infrastructure', 'grid', 'attack']): name = "Атаки на инфраструктуру"
    elif any(w in top_words for w in ['autonomous', 'car', 'tesla']): name = "Автономные системы и риски"
    cluster_names[c] = name

df_ai['typology_name'] = df_ai['cluster'].map(cluster_names)
print("\nРаспределение по типам:")
print(df_ai['typology_name'].value_counts())

In [ ]:
# 8. ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=umap_emb[:, 0], y=umap_emb[:, 1],
    hue=df_ai['typology_name'],
    palette='tab10',
    s=60, alpha=0.7
)
plt.title(f'Кластеры BERT+UMAP+HDBSCAN (Silhouette: {score:.3f})')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
#WORDCLOUD

from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter

#Расширенные стоп-слова
extra_stopwords = {
    'said', 'one', 'new', 'back', 'also', 'would', 'could', 'may', 'might',
    'has', 'have', 'had', 'is', 'was', 'were', 'be', 'been', 'that', 'this',
    'with', 'from', 'they', 'their', 'them', 'will', 'into', 'over', 'after',
    'just', 'now', 'more', 'like', 'can', 'do', 'not', 'but', 'are', 'you'
}

def get_filtered_text(typ):
    subset = df_ai[df_ai['typology_name'] == typ]

    texts = []
    for item in subset['clean_text_str']:
        if isinstance(item, list):
            texts.extend(item)
        elif isinstance(item, str):
            texts.append(item)

    full_text = ' '.join(texts).lower()

    words = full_text.split()
    filtered = [w for w in words if w not in extra_stopwords and len(w) > 3]
    return ' '.join(filtered)

#Генерация WordCloud
unique_types = df_ai['typology_name'].unique()
n_types = len(unique_types)

fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 6))
if n_types == 1:
    axes = [axes]

for idx, typ in enumerate(unique_types):
    text = get_filtered_text(typ)

    if text.strip():
        wc = WordCloud(
            width=600, height=400,
            background_color='white',
            max_words=60,
            contour_width=1,
        ).generate(text)

        axes[idx].imshow(wc, interpolation='bilinear')
        axes[idx].set_title(f"{typ}\n(после фильтрации)", fontsize=13, pad=15)
    else:
        axes[idx].text(0.5, 0.5, 'Нет данных', ha='center', va='center', fontsize=14)
        axes[idx].set_title(typ, fontsize=13)

    axes[idx].axis('off')

plt.suptitle('Облака слов по типам угроз ИИ\n(удалены лишние слова)',
             fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# 9. ОБУЧЕНИЕ КЛАССИФИКАТОРА
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,3), sublinear_tf=True)
X = vectorizer.fit_transform(df_ai['clean_text_str'])
y = df_ai['typology_name']

smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("\nКачество модели:")
print(classification_report(y_test, model.predict(X_test)))

joblib.dump(model, 'classifier.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("Модель сохранена")

In [ ]:
# 10. ПРЕДСКАЗАНИЕ
def predict(text):
    clean = ' '.join(preprocess_text(text))
    vec = vectorizer.transform([clean])
    pred = model.predict(vec)[0]
    conf = model.predict_proba(vec).max()
    return pred, conf

print(predict("The autonomous bus had an accident on its first trip on the route"))
print(predict("A fake video with a Mask singing about the Earth in a porthole blew up the Network"))
print(predict("An explosion occurred at a plant in Sterlitamak after a UAV attack"))
print(predict("businesses began to become disillusioned with artificial intelligence"))

In [ ]:
df_ai.shape

In [ ]:
# 11. ВИЗУАЛИЗАЦИЯ ТРЕНДОВ И КАРТЫ: ПОЛНЫЙ ОБЪЁМ (7088 новостей, 8814 упоминаний)

# Подготовка данных
df_full = df_ai.copy()

start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)

def random_date():
    delta = end_date - start_date
    days = random.randint(0, delta.days)
    return start_date + timedelta(days=days)

mask_no_date = df_full['date_published'].isna()
df_full.loc[mask_no_date, 'date_published'] = [random_date() for _ in range(mask_no_date.sum())]

# Обновляем year
df_full['date_published'] = pd.to_datetime(df_full['date_published'], errors='coerce')
df_full['year'] = df_full['date_published'].dt.year

print(f"Всего новостей с датами: {len(df_full)}")
print("Годы:", sorted(df_full['year'].unique()))

# Explode по странам
df_exploded = df_full.explode('countries').dropna(subset=['countries'])

print(f"Упоминаний стран: {len(df_exploded)}")
print(df_exploded['countries'].value_counts())

In [ ]:
#Пересоздаём корректную дату
df_ai["date_published"] = pd.to_datetime(df_ai["date_published"], utc=True).dt.tz_convert(None)

#Пересоздаём год
df_ai["year"] = df_ai["date_published"].dt.year

#Переносим год в exploded
df_exploded = df_ai.copy()
df_exploded["countries"] = df_ai["countries"].apply(lambda x: x if isinstance(x, list) else [])
df_exploded = df_exploded.explode("countries")

#Группировка
trend = (
    df_exploded.groupby(["year", "countries"])
    .size()
    .reset_index(name="count")
)

print("Min year =", trend["year"].min())
print("Max year =", trend["year"].max())


In [ ]:
# График по трендам (Сравнение динамики США, Китая, России)

trend_norm = trend.copy()

trend_norm["count_norm"] = trend_norm.groupby("countries")["count"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

filtered_trend_norm = trend_norm[trend_norm['countries'].isin(['USA', 'China', 'Russia'])]

plt.figure(figsize=(14, 7))

for country in filtered_trend_norm["countries"].unique():
    subset = filtered_trend_norm[filtered_trend_norm["countries"] == country]
    plt.plot(subset["year"], subset["count_norm"], marker="o", label=country)

plt.title("Сравнение динамики ИИ-угроз: США, Китай, Россия", fontsize=16)
plt.xlabel("Год")
plt.ylabel("Интенсивность (нормализованная)")
plt.grid(True, alpha=0.3)

min_year = int(filtered_trend_norm['year'].min())
max_year = int(filtered_trend_norm['year'].max())

plt.xlim(2011, 2026)
plt.xticks(range(2011, 2026))

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#HEATMAP: СТРАНА × ТИП ИИ Угрозы
heatmap_data = df_exploded.groupby(['countries', 'typology_name']).size().unstack(fill_value=0)
heatmap_data = heatmap_data.loc[['USA', 'China', 'Russia']]

desired_order = [
    "Атаки на инфраструктуру",
    "Deepfake & медиафальсификации",
    "Автономные системы и риски",
    "Другое"
]
heatmap_data = heatmap_data.reindex(columns=[c for c in desired_order if c in heatmap_data.columns])

plt.figure(figsize=(12, 5))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='d',
    cmap='YlOrRd', linewidths=1, linecolor='black',
    cbar_kws={'label': 'Количество упоминаний'}
)
plt.title('Heatmap: Типы ИИ-угроз по странам (8814 упоминаний)', fontsize=16, fontweight='bold')
plt.ylabel('Страна')
plt.xlabel('Тип')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#ИНТЕРАКТИВНАЯ КАРТА
import folium
from folium.plugins import HeatMap
import pandas as pd

m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')

heat_data = []

coords = {
    'USA': [39.8283, -98.5795],
    'China': [35.8617, 104.1954],
    'Russia': [61.5240, 105.3188]
}

for country in ['USA', 'China', 'Russia']:
    if 'heatmap_data' in locals() and isinstance(heatmap_data, pd.DataFrame):
        if country in heatmap_data.index:
            count = heatmap_data.loc[country].sum()
            lat, lon = coords[country]
            heat_data.append([lat, lon, int(count)]) # Convert count to int
HeatMap(heat_data, radius=25, blur=15).add_to(m)


for country in ['USA', 'China', 'Russia']:
    lat, lon = coords[country]
    total = heatmap_data.loc[country].sum()

    popup_html = f"""
    <div style="font-family: Arial; width: 280px;">
        <h4>{country}</h4>
        <p><b>Всего:</b> {int(total)}</p>
        <hr>
    """
    if 'heatmap_data' in locals() and isinstance(heatmap_data, pd.DataFrame):
        if country in heatmap_data.index:
            for typ, count in heatmap_data.loc[country].items():
                popup_html += f"""
                <div style="display: flex; justify-content: space-between;">
                    <span>{typ}</span>
                    <span style="font-weight: bold;">{int(count)}</span>
                </div>
                """
    popup_html += "</div>"

    folium.CircleMarker(
        [lat, lon],
        radius=10 + int(total) / 50, # Convert total to int
        color='black',
        fill=True,
        fill_color='red',
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{country}: {int(total)}" # Convert total to int
    ).add_to(m)

m.save('interactive_map.html')
print("\nИнтерактивная карта сохранена: 'interactive_map.html'")
display(m)